# 초구 예측 — model_02

- 1단계 항목: 위치 기준점 바꾸기 (기준점: 반감기 적용 or 직전 시즌만)
  - 기준점 = 위치를 당기는 출발점, 리그 구종×타자 손 평균 (model_01은 지난 시즌 전부를 같은 무게로)
  - 이유: 베이스라인에서 2020 초구가 이전보다 존 안에 더 많이 들어왔는데 옛날 리그 비율 때문에 따라가지 못함 (Chase 치우침)
- 바꾸는 것: 기준점 후보 3가지 + 위치 파라미터(반감기, k_type, k_hand)만 다시 고름, 구종은 model_01 값 고정
- 비교 대상: model_01 (2019로 고르고, 2020은 compare 부트스트랩으로 확인)

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc  

MODEL = "model_02"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 파라미터

- 구종: model_01 값 고정
- 위치: model_01 값에서 시작 → 아래 3-2에서 기준점마다 다시 고르고 1등으로 덮어씀

In [2]:
# ① 구종 (model_01, 고정): 반감기(공 수), 리그 쪽 당김, 타자 손 쪽 당김
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 25.0, 5.48, 5.48
# ② 위치 (model_01에서 시작, 다시 고름): 반감기(공 수), 투수×구종 / 투수×구종×타자 손 당김
LOC_HL, K_TYPE, K_HAND = 35.4, 54.8, 17.3
# ② 위치 기준점: "all" 지난 시즌 전부 (model_01) / "last" 직전 시즌만 / "decay" 전날까지 반감기 가중 (LOC_REF_HL = 반감기, 공 수)
LOC_REF, LOC_REF_HL = "all", None

## 2. ① 구종 (model_01과 같음)

In [3]:
def league_type(year):
    return fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()

def type_online(q, year, half_life, alpha, hand_k):
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, half_life), q, ["pitcher_code"], CLASSES)
    P = fc.shrink(S, N[:, :1], league_type(year), alpha)
    if hand_k:
        keys = ["pitcher_code", "same_side"]
        S2, N2 = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life), q, keys, CLASSES)
        P = fc.shrink(S2, N2[:, :1], P, hand_k)
    return P

## 3. ② 위치

- model_01과 같은 순서(기준점 → 투수×구종 → 투수×구종×타자 손)로 당김, 기준점만 3가지로
  - `all`: 지난 시즌 전부 같은 무게 (model_01)
  - `last`: 직전 시즌만 -> 오래된 시즌 빼기
  - `decay`: 전날까지 구종×손별 반감기 가중 -> 시즌 중에도 갱신, 최근 리그 경향을 따라감 (다른 투수의 트래킹 값도 전날까지만이라 정보 기준에 맞음)

In [5]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 → 정규분포 (높이는 그 타자 존으로 cm 환산)
    return fc.zone_from_gauss(g, qt), g  # 정규분포를 24칸 위에서 적분 (좌우·높이 독립 가정)

### 3-1. 진단: model_01 위치의 H/S/C 치우침

- 실제 구종 기준 위치 예측의 H/S/C 확률 평균 vs 실제 비율, 연도·월별
- 예측 − 실제가 +이면 그 묶음을 너무 많이 예측

In [6]:
def hsc_bias(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 월별 H/S/C 예측 평균 − 실제 비율 (실제 구종 기준)
    g = fc.gauss(loc_online(q, year, half_life, k_type, k_hand, ref, ref_hl), q)
    P = fc.zone_from_gauss(g, q) @ fc.GROUP2  # 24칸 확률 → H/S/C (n, 3)
    A = pd.get_dummies(q["zone_group"]).reindex(columns=fc.HSC, fill_value=0).to_numpy(float)  # 실제 H/S/C 원-핫
    d = pd.DataFrame(P - A, columns=["H 차이", "S 차이", "C 차이"])
    d["월"] = q["game_date"].dt.month.to_numpy()
    t = d.groupby("월").mean()
    t.loc["전체"] = d.drop(columns="월").mean()
    t["공수"] = d.groupby("월").size().reindex(t.index).fillna(len(d)).astype(int)
    return t

M01_LOC = (35.4, 54.8, 17.3)  # model_01 위치 파라미터
for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    print(year, "· model_01 (기준점 all)")
    display(hsc_bias(q, year, *M01_LOC).round(3))

2019 · model_01 (기준점 all)


,H 차이,S 차이,C 차이,공수
월,,,,
3,-0.025,-0.022,0.048,703
4,-0.018,-0.019,0.037,1865
5,-0.020,-0.017,0.037,2403
6,-0.020,-0.022,0.042,2493
7,-0.005,-0.004,0.009,1791
8,-0.017,-0.019,0.036,2362
9,-0.038,-0.005,0.042,1794
10,0.058,0.025,-0.083,47
전체,-0.020,-0.015,0.035,13458


2020 · model_01 (기준점 all)


,H 차이,S 차이,C 차이,공수
월,,,,
5,-0.015,-0.016,0.030,2264
6,-0.012,-0.002,0.013,2379
7,-0.018,-0.007,0.025,2233
8,0.007,-0.033,0.026,1985
9,-0.003,-0.003,0.006,1982
10,-0.031,-0.007,0.038,1772
전체,-0.012,-0.011,0.023,12615


### 🔍 model_01도 Chase 너무 많이 예측
#### C 차이 2019 +0.035, 2020 +0.023 (H·S는 그만큼 적게)
- 거의 모든 달에서 같은 방향 (2019 10월만 반대, 공 47개)
- 2020이 2019보다 작음 -> "옛날 리그 비율 때문"이라면 2020이 더 커야 함 -> 기준점만의 문제는 아닐 수 있음

### 3-2. 기준점 후보마다 위치 파라미터 선택 (2019)

- 기준점마다 반감기·k_type·k_hand를 다시 고름 (기준점이 바뀌면 당기는 강도의 최적값도 바뀜), `decay`는 기준점 반감기도 함께
- 거친 격자는 model_01과 같은 후보 -> `all`은 model_01 선택을 그대로 재현 (대조군)
- 기준: 2019 24칸 log loss (실제 구종 기준), H/S/C log loss·좌표 NLL은 참고

In [7]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN → None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로"""
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN → None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

In [8]:
def score_loc(half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 파라미터 한 조합의 2019 채점: 24칸 log loss·top1·H/S/C + 좌표 NLL
    g = fc.gauss(loc_online(q_val, fc.VAL_YEAR, half_life, k_type, k_hand, ref, ref_hl), q_val)  # 실제 구종 기준 정규분포
    z = fc.zone_metrics(fc.zone_from_gauss(g, q_val), y_zone[val])
    c = fc.coord_metrics(g, q_val)
    return {"logloss": z["logloss"], "top1": z["top1"], "H/S/C logloss": z["H/S/C logloss"],
            "H/S/C 정확도": z["H/S/C 정확도"], "좌표 NLL": c["NLL"], "평균거리 cm": c["평균거리 cm"]}

GRID_LOC = {"half_life": [None, 1000, 300, 100, 50],  # model_01과 같은 거친 격자
            "k_type": [10, 30, 100, 300, 1000, 3000],
            "k_hand": [10, 30, 100, 300, 1000, 3000]}
REF_HL_GRID = [30, 100, 300, 1000, 3000, 10000]  # decay 기준점 반감기 (구종×손 그룹의 공 수, 직구×같은 손 초구가 한 시즌 약 3,500개)

def select_loc(ref):  # 기준점 하나에 대해 거친 격자 → 세밀한 격자, 반환: 1등 파라미터 dict, 거친 표, 세밀한 표, 세밀한 후보
    grid = {**GRID_LOC, **({"ref_hl": REF_HL_GRID} if ref == "decay" else {})}
    score = lambda **p: score_loc(ref=ref, **p)
    res = grid_search(score, grid)
    best = best_of(res, grid)
    grid_f = {k: finer(grid[k], best[k]) for k in grid}  # 1등 주변 세밀한 후보
    res_f = grid_search(score, grid_f)
    return {"ref": ref, "ref_hl": None, **best_of(res_f, grid_f)}, res, res_f, grid_f

In [9]:
sel = {ref: select_loc(ref) for ref in ["all", "last", "decay"]}  # 기준점마다 선택 (decay는 반감기 후보만큼 오래 걸림)

rows = {}
for ref, (b, res, res_f, _) in sel.items():
    rows[ref] = {**{k: b[k] for k in ["ref_hl", "half_life", "k_type", "k_hand"]}, **res_f.iloc[0][["logloss", "top1", "H/S/C logloss", "좌표 NLL"]].to_dict()}
ref_tab = pd.DataFrame(rows).T
ref_tab["model_01 대비"] = ref_tab["logloss"] - ref_tab.loc["all", "logloss"]  # all = model_01 (대조군)
display(ref_tab)
for k, s in marginal(sel["decay"][2], sel["decay"][3]).items():
    print("decay ·", k, s.round(4).to_dict())  # decay 세밀한 격자의 파라미터별 민감도

,ref_hl,half_life,k_type,k_hand,logloss,top1,H/S/C logloss,좌표 NLL,model_01 대비
all,NaN,35.4,54.8,17.3,2.927023,0.144821,1.075246,9.261045,0.000000
last,NaN,35.4,100.0,17.3,2.926156,0.144449,1.074967,9.259472,-0.000867
decay,173.0,35.4,548.0,17.3,2.922206,0.145564,1.074513,9.252413,-0.004817


decay · half_life {25.0: 2.9223, 35.4: 2.9222, 50.0: 2.9229, 70.7: 2.9241, 100.0: 2.9257}
decay · k_type {100.0: 2.9223, 173.0: 2.9223, 300.0: 2.9222, 548.0: 2.9222, 1000.0: 2.9222}
decay · k_hand {10.0: 2.9233, 17.3: 2.9222, 30.0: 2.9223, 54.8: 2.9237, 100.0: 2.9257}
decay · ref_hl {30.0: 2.9236, 54.8: 2.9227, 100.0: 2.9222, 173.0: 2.9222, 300.0: 2.9225}


### 🥇1등🥇
#### 기준점 decay (반감기 173구), 반감 35.4, k_type 548, k_hand 17.3
#### -> 2019 24칸 log loss 2.9222 (model_01 대비 −0.0048)
- last(직전 시즌만)는 −0.0009로 거의 차이 없음
- decay 반감기: 100–173구가 좋음 (30구 2.9236, 300구 2.9225)
  - 구종×손 그룹의 리그 공 173개 -> 직구처럼 많은 그룹은 최근 며칠~1–2주, 드문 구종은 더 긴 기간
- k_type: 100–1000에서 2.9222–2.9223으로 평평 -> 기준점이 최근 리그를 따라가면 투수×구종 단계는 거의 기준점 쪽에 붙음 (투수 개인 차이는 k_hand 단계에서)
- half_life, k_hand는 model_01과 같은 값 (35.4, 17.3)

In [10]:
# 선택: 2019 24칸 log loss가 가장 낮은 기준점 (실전 예측은 이 값 사용)
best_ref = ref_tab["logloss"].astype(float).idxmin()
b = sel[best_ref][0]
LOC_REF, LOC_REF_HL = b["ref"], b["ref_hl"]
LOC_HL, K_TYPE, K_HAND = b["half_life"], b["k_type"], b["k_hand"]
print(f"선택: 기준점 {LOC_REF} (반감기 {LOC_REF_HL}), half_life={LOC_HL}, k_type={K_TYPE}, k_hand={K_HAND}")

선택: 기준점 decay (반감기 173.0), half_life=35.4, k_type=548.0, k_hand=17.3


### 3-3. 진단: 기준점별 C 치우침 (적용 후 같은 그룹으로 확인)

In [11]:
rows = {}
for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    for ref, (b, _, _, _) in sel.items():
        rows[(year, ref)] = hsc_bias(q, year, b["half_life"], b["k_type"], b["k_hand"], ref, b["ref_hl"])["C 차이"]  # 기준점마다 자기 파라미터로
display(pd.concat(rows, axis=1).round(3))  # 열 = (연도, 기준점), 행 = 월 + 전체

2019                 2020              
      all   last  decay    all   last  decay
월                                           
3   0.048  0.045  0.045    NaN    NaN    NaN
4   0.037  0.035  0.034    NaN    NaN    NaN
5   0.037  0.036  0.034  0.030  0.025  0.024
6   0.042  0.041  0.036  0.013  0.009  0.009
7   0.009  0.009  0.002  0.025  0.021  0.022
8   0.036  0.036  0.032  0.026  0.022  0.023
9   0.042  0.042  0.036  0.006  0.002  0.003
10 -0.083 -0.084 -0.091  0.038  0.034  0.034
전체  0.035  0.034  0.030  0.023  0.019  0.019

### 🔍 C 치우침은 조금만 줄어듦
#### 전체 C 차이: 2019 0.035 → 0.030, 2020 0.023 → 0.019 (decay)
- last, decay 모두 2020에서 −0.004 정도로 비슷
- 월별로도 모든 달에서 비슷하게 조금씩 줄어듦 -> 기준점은 치우침의 일부만 설명
- 남은 +0.02–0.03은 기준점이 아닌 다른 원인 -> 7장에서 확인: 주로 좌우·높이 독립 가정 (약 0.01), 정규분포 모양은 작음 (약 0.006), 2019는 리그 추세도

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_01과 같음, 위치에 기준점(`LOC_REF`, `LOC_REF_HL`)만 추가

In [12]:
def predict(q, year):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9)
    Pt = type_online(q, year, TYPE_HL, TYPE_ALPHA, TYPE_HAND_K)

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [13]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR), predict(q_test, fc.TEST_YEAR)], ignore_index=True)
print(pp.shape)

(26073, 48)


## 5. 평가

- 채점은 `fc.evaluate`, model_01·베이스라인과 나란히
- 2020은 확인용 → 2020을 보고 기준점·파라미터 다시 고르지 않음

In [14]:
def eval_years(pp_):
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m01_sum = eval_years(pd.read_parquet("results/model_01/per_pitch.parquet"))  # model_01 공별 예측
base_sum = eval_years(pd.read_parquet("results/baseline/per_pitch.parquet"))  # 베이스라인 공별 예측
display(pd.concat({MODEL: summary, "model_01": m01_sum, "차이 (model_01 대비)": summary - m01_sum}, axis=1).round(4))

model_02          model_01          차이 (model_01 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0982   1.0956   1.0982   1.0956           0.0000  0.0000
구종 top1               0.5258   0.5233   0.5258   0.5233           0.0000  0.0000
구종 top2               0.8157   0.8162   0.8157   0.8162           0.0000  0.0000
구종 top3               0.9331   0.9359   0.9331   0.9359           0.0000  0.0000
구종 macro F1           0.2476   0.2857   0.2476   0.2857           0.0000  0.0000
24칸 logloss (실전)      2.9756   2.9698   2.9780   2.9708          -0.0024 -0.0011
24칸 top1              0.1286   0.1331   0.1289   0.1339          -0.0003 -0.0008
24칸 top3              0.2865   0.2859   0.2841   0.2850           0.0025  0.0009
H/S/C logloss         1.0766   1.0778   1.0771   1.0783          -0.0006 -0.0005
H/S/C 정확도             0.3931   0.3874   0.3947   0.3902          -0.0016 -0.0029
24칸 logloss (실제 구종)   2.9222   2.9225   2.9270   2.9226          -0.0048 -0.0001
구종×칸 logloss          4.0204   4.0177   4.0252   4.0182          -0.0048 -0.0005
좌표 평균거리 cm           32.7399  31.8840  32.7934  31.9009          -0.0535 -0.0169
좌표 NLL                9.3194   9.2689   9.3235   9.2708          -0.0041 -0.0019

- 위치만 바뀜 -> 구종 지표는 model_01과 같음
- 2020 24칸 log loss 실전 2.9708 → 2.9698 (−0.0011), 실제 구종 2.9226 → 2.9225 (−0.0001)
  - 2019 개선(−0.0048)이 2020에는 거의 안 이어짐
  - 최근 리그 경향을 따라가는 효과가 2019에만 컸을 가능성 (2019 시즌 중 리그 전체 위치 변화?)
- 좌표 NLL −0.002, 평균거리 −0.02cm, H/S/C 정확도 −0.003
- 경기 단위 부트스트랩 (model_02 − model_01, 2020)
  - 좋아짐: 실전 24칸 −0.0011 (95% −0.0018 ~ −0.0004), 좌표 NLL −0.0019, 좌표 거리 −0.017cm
  - 차이 불확실: 구종×칸 −0.0005 (−0.0033 ~ +0.0032)
- -> 방향은 일관되게 좋아졌지만 크기는 작음 (model_01이 베이스라인 대비 −0.008이었던 것의 1/8)

## 6. 저장

- `results/model_02/` → `compare.ipynb` 실행하면 자동 포함 (부트스트랩 판정은 compare에서)

In [15]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_02 ['per_pitch.parquet', 'summary.csv']


## 7. 진단: C 치우침이 기준점으로 조금만 줄어든 이유

- 3-3: decay로 C 치우침 2019 0.035 → 0.030, 2020 0.023 → 0.019 -> 기준점은 일부만 설명
- 확인할 것 (model_02 위치 설정, 실제 구종 기준)
  1. 24칸 계산 방식별 치우침: 정규분포 (좌우·높이 독립, 지금) / 정규분포 (공분산 반영) / 칸 비율 (정규분포 없이)
  2. 어느 축에서: 좌우로 바깥 vs 높이로 바깥
  3. 정규분포 모양이 맞는지: 표준화 잔차 분포
  4. 시간 흐름: 연도·월별 실제 초구 C 비율

In [16]:
LOC_M02 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL)  # 위 3-2에서 고른 위치 설정
rng = np.random.default_rng(0)  # 공분산 반영 시뮬레이션용 (결과 재현)

def zone_parts(q, year, n_draw=400):  # 공마다 H/S/C 확률 3가지 방식 (n, 3) + 정규분포 g
    M = loc_online(q, year, **LOC_M02)  # 모멘트 5개 + 24칸 비율
    g = fc.gauss(M, q)
    mx, mz, vx, vz, cv = g
    # 공분산 반영: 2차원 정규분포에서 n_draw개 뽑아 칸 세기 (촐레스키 분해로 상관 있는 표본)
    L11 = np.sqrt(vx); L21 = cv / L11; L22 = np.sqrt(np.maximum(vz - L21 ** 2, 1e-9))
    e1, e2 = rng.standard_normal((len(q), n_draw)), rng.standard_normal((len(q), n_draw))
    sx, sz = mx[:, None] + L11[:, None] * e1, mz[:, None] + L21[:, None] * e1 + L22[:, None] * e2
    ez = fc.z_edges(q["TOP_SZ_CM"].to_numpy(), q["BOTTOM_SZ_CM"].to_numpy())
    cell36 = (sz[:, :, None] >= ez[:, None, :]).sum(2) * 6 + np.digitize(sx, fc.X_EDGES)  # 표본마다 36칸 번호
    grp = fc.ZONE_GROUP[[fc.ZONE_IDX[c] for c in fc.CELL_NAMES]][cell36]  # 표본마다 H/S/C 글자
    return {"정규분포 (독립)": fc.zone_from_gauss(g, q) @ fc.GROUP2,
            "정규분포 (공분산 반영)": np.stack([(grp == h).mean(1) for h in fc.HSC], 1),
            "칸 비율": M[:, len(MOM):] @ fc.GROUP2}, g

parts = {year: zone_parts(q, year) for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]}

In [17]:
# 1. 방식별 C 예측 평균 − 실제
rows = {}
for (year, (P, _)), q in zip(parts.items(), [q_val, q_test]):
    actual = (q["zone_group"] == "C").mean()
    rows[year] = {"실제 C 비율": actual, **{k + " − 실제": v[:, 2].mean() - actual for k, v in P.items()}}
display(pd.DataFrame(rows).round(4))

,2019,2020
실제 C 비율,0.3925,0.3853
정규분포 (독립) − 실제,0.0302,0.0185
정규분포 (공분산 반영) − 실제,0.0198,0.0093
칸 비율 − 실제,0.0138,0.0032


### 🔍 C 치우침은 원인 세 가지가 겹침
#### C 예측 − 실제: 독립 정규분포 2019 +0.030 / 2020 +0.019
- 공분산 반영하면 +0.020 / +0.009 -> 좌우·높이 독립 가정이 약 0.01 (가장 큼)
  - 좌표 NLL에는 이미 공분산을 쓰는데 24칸 적분에서만 버리고 있었음
- 칸 비율은 +0.014 / +0.003 -> 정규분포 모양 자체는 약 0.006
- 칸 비율에도 2019에 +0.014가 남음 -> 계산 방식이 아닌 다른 원인

In [18]:
# 2. 축별: 정규분포(독립)에서 좌우·높이 각 구간에 들어갈 확률 평균 vs 실제 비율
from scipy.special import ndtr

def axis_bias(q, g):  # 바깥(Chase)·Shadow 띠 구간별 예측 − 실제
    mx, mz, vx, vz, _ = g
    x, z = q["x"].to_numpy(), q["CROSS_PLATE_Z_CM"].to_numpy()
    ez = fc.z_edges(q["TOP_SZ_CM"].to_numpy(), q["BOTTOM_SZ_CM"].to_numpy())
    inf = np.full(len(q), np.inf)
    Px = np.diff(ndtr((np.r_[-np.inf, fc.X_EDGES, np.inf][None, :] - mx[:, None]) / np.sqrt(vx)[:, None]), axis=1)  # 좌우 6구간 확률
    Pz = np.diff(ndtr((np.column_stack([-inf, ez, inf]) - mz[:, None]) / np.sqrt(vz)[:, None]), axis=1)  # 높이 6구간 확률
    xi, zi = np.digitize(x, fc.X_EDGES), (z[:, None] >= ez).sum(1)  # 실제 구간 번호
    names_ = {("x", 0): "좌우 바깥 (밖)", ("x", 5): "좌우 바깥 (몸)", ("z", 0): "높이 바깥 (낮)", ("z", 5): "높이 바깥 (높)",
              ("x", 1): "좌우 Shadow 띠 (밖)", ("x", 4): "좌우 Shadow 띠 (몸)", ("z", 1): "높이 Shadow 띠 (하)", ("z", 4): "높이 Shadow 띠 (상)"}
    return pd.Series({nm: (Px if ax == "x" else Pz)[:, j].mean() - ((xi if ax == "x" else zi) == j).mean() for (ax, j), nm in names_.items()})

display(pd.DataFrame({year: axis_bias(q, parts[year][1]) for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]}).round(4))

,2019,2020
좌우 바깥 (밖),0.0100,0.0036
좌우 바깥 (몸),0.0015,0.0008
높이 바깥 (낮),0.0235,0.0067
높이 바깥 (높),-0.0081,0.0002
좌우 Shadow 띠 (밖),-0.0085,-0.0036
좌우 Shadow 띠 (몸),0.0020,-0.0007
높이 Shadow 띠 (하),0.0106,0.0002
높이 Shadow 띠 (상),-0.0023,0.0016


### 🔍 2019 치우침은 주로 낮게 빠지는 공
- 2019 높이 바깥 (낮) +0.024, 좌우 바깥 (밖) +0.010, 나머지는 ±0.01 이내
- 2020은 모든 구간 0.007 이하

In [19]:
# 3. 정규분포 모양: 실제 공이 예측 중심에서 (예측) 표준편차 몇 개만큼 떨어졌나, 정규분포라면 첫 열 비율
bins = [0, 0.5, 1, 1.5, 2, 3, np.inf]
cols = {"정규분포": np.diff(2 * ndtr(np.array(bins[:-1] + [50])) - 1)}  # |z| 구간별 정규분포 확률
for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    mx, mz, vx, vz, _ = parts[year][1]
    for ax, r in [("좌우", (q["x"].to_numpy() - mx) / np.sqrt(vx)), ("높이", (q["CROSS_PLATE_Z_CM"].to_numpy() - mz) / np.sqrt(vz))]:
        cols[f"{year} {ax}"] = pd.Series(pd.cut(np.abs(r), bins, right=False)).value_counts(normalize=True, sort=False).to_numpy()
display(pd.DataFrame(cols, index=[f"{a}–{b}" for a, b in zip(bins[:-1], bins[1:])]).round(4))

,정규분포,2019 좌우,2019 높이,2020 좌우,2020 높이
0–0.5,0.3829,0.3866,0.3990,0.3857,0.3865
0.5–1,0.2998,0.3119,0.2992,0.2985,0.3046
1–1.5,0.1837,0.1757,0.1745,0.1856,0.1753
1.5–2,0.0881,0.0846,0.0811,0.0850,0.0843
2–3,0.0428,0.0377,0.0418,0.0417,0.0440
3–inf,0.0027,0.0035,0.0044,0.0036,0.0052


### 🔍 정규분포 모양은 거의 맞음
- 실제 공의 |표준화 잔차| 구간 비율이 정규분포와 거의 같음 (0–0.5: 0.39–0.40 vs 0.38, 3 이상: 0.003–0.005 vs 0.003) -> 꼬리가 두껍거나 가운데가 뾰족한 문제는 아님, 정규분포 모양 때문인 몫(약 0.006)은 작음
- 24칸 log loss는 정규분포가 칸 비율보다 훨씬 좋음 (2019 2.922 vs 2.981) -> 정규분포 방식 유지

In [20]:
# 4. 시간 흐름: 연도·월별 실제 초구 C 비율 (전체 초구)
c_share = fp.assign(C=(fp["zone_group"] == "C").astype(float), 월=fp["game_date"].dt.month)
t = c_share.pivot_table(index="월", columns="GYEAR", values="C", aggfunc="mean")
t.loc["전체"] = c_share.groupby("GYEAR")["C"].mean()
display(t.round(3))

GYEAR,2017,2018,2019,2020
월,,,,
3,0.600,0.439,0.397,NaN
4,0.415,0.415,0.398,NaN
5,0.421,0.410,0.395,0.379
6,0.441,0.421,0.390,0.397
7,0.462,0.427,0.413,0.382
8,0.414,0.433,0.382,0.378
9,0.436,0.406,0.376,0.405
10,0.451,0.431,0.489,0.368
전체,0.431,0.420,0.392,0.385


### 🔍 실제 초구 C 비율 시즌마다 줄어듦
#### 0.431 (2017) → 0.420 (2018) → 0.392 (2019) → 0.385 (2020)
- 2019에 크게 줄어듦 (−0.028) -> 과거 기록이 늦게 따라가 칸 비율도 2019에 C를 +0.014 많이 예측
  - 2019 공인구 변경?
- 2020은 −0.007로 작음 -> 칸 비율 치우침 +0.003
- model_02 결과와 맞음: decay 기준점이 2019에서 크게 좋아지고 2020은 거의 그대로였던 이유 = 2019에만 따라갈 변화가 컸음

### 결론
| 원인 | 크기 | 처리 |
| --- | --- | --- |
| 좌우·높이 독립 가정 | 약 0.01 | model_03에서 24칸 적분에 공분산 반영 |
| 정규분포 모양 | 약 0.006 | 지금은 유지 (잔차가 정규분포와 거의 같음), model_03 뒤 남은 치우침 다시 확인 |
| 리그 추세 (2019) | 2019만 약 0.014 | 모델 결함 아님, decay 기준점이 대응, 2020은 거의 없음 |